In [1]:
# general imports
import pandas as pd

# MMAI Examples

## 1. Trial summarization walkthrough


### Read in data

In [2]:
trials_path = "data/scheduled__2025-09-04T230000+0000.trials_for_summarize.csv"
trials = pd.read_csv(trials_path)

### Local mode

`summarize_trials(...)` uses the local in-memory vLLM backend by default. Run this cell if you want local mode.


In [ ]:
from matchminer_ai.trials import summarize_trials

trial_results, metadata, qc_report = summarize_trials(
    trials, return_metadata=True, return_qc=True
)

### Remote mode

Run this cell instead of the local-mode cell if you want to send summarization requests to an existing OpenAI-compatible endpoint. If you do not already have a vLLM server running, `start_vllm_server(...)` can start one from the config values in`. The same local/remote pattern applies to patient summarization in the next section.


In [ ]:
import os
from matchminer_ai import load_preset
from matchminer_ai.trials import summarize_trials
# from matchminer_ai.llm.vllm_server import start_vllm_server

os.environ["OPENAI_API_KEY"] = "not-needed"

# Optional: start a local vLLM server if you do not already have one.
# process = start_vllm_server(task="patient")

config = load_preset("default")
config.remote["enabled"] = True
config.remote["server_urls"] = ["http://localhost:8000/v1"]

trial_results, metadata, qc_report = summarize_trials(
    trials, config=config, return_metadata=True, return_qc=True
)

### View results

In [5]:
trial_results.head()

,trial_id,clinical_space_number,clinical_space_summary,general_exclusion_criteria,space_trial_id
0,NCT03319901,1,Age range allowed: ≥60 years. Sex allowed: All...,"History of pneumonia/pneumonitis,\nUncontrolle...",NCT03319901-1
1,NCT03319901,2,Age range allowed: ≥18 years. Sex allowed: All...,"History of pneumonia/pneumonitis,\nUncontrolle...",NCT03319901-2
2,NCT04792489,1,Age range allowed: ≥18 years. Sex allowed: Mal...,* Inability to give informed consent \n* Curr...,NCT04792489-1
3,NCT04792489,2,Age range allowed: ≥18 years. Sex allowed: Mal...,* Inability to give informed consent \n* Curr...,NCT04792489-2
4,NCT04792489,3,Age range allowed: ≥18 years. Sex allowed: Mal...,* Inability to give informed consent \n* Curr...,NCT04792489-3


Metadata contains a snapshot of the config used and metadata on the model used. 

In [6]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 10000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9},
   'patient': {'max_model_len': 100000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'max_concurrent_requests': 16,
   'request_timeout': 600,
   'max_retries': 6,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'model_name': 'openai/gpt-oss-120b',
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.3},
   'prompt_files': {'primer': 'trial.user.primer.txt',
    'question': 'trial.user.question.txt'},
   'reasoning_marker': 'assistantfinal',
   'boilerplate_marker': '\\n.*Boilerplate.*\\n'},
  'patient': {'model_name': 'openai/gpt-oss-120b',
   'chunk_size': 10000,
   'chunk_overla

QC report on the trial summarization run

In [7]:
qc_report

,metric,value,denominator,percent,ids
0,trials_missing_in_output,3.0,470.0,0.638298,"[NCT04771078, NCT06137118, NCT06153251]"
1,trials_truncated_llm_response,1.0,470.0,0.212766,[NCT04771572]
2,spaces_exceed_embedding_token_limit,0.0,1444.0,0.000000,[]
3,spaces_per_trial_min,1.0,NaN,NaN,[]
4,spaces_per_trial_median,2.0,NaN,NaN,[]
5,spaces_per_trial_max,33.0,NaN,NaN,[]
6,trials_with_non_distinct_spaces,4.0,467.0,0.856531,"[NCT04777994, NCT05536141, NCT05827614, NCT065..."
7,spaces_dropped_missing_keyword:Age,0.0,1449.0,0.000000,[]
8,spaces_dropped_missing_keyword:Sex,0.0,1449.0,0.000000,[]
9,spaces_dropped_missing_keyword:Cancer type,0.0,1449.0,0.000000,[]


Save checked-in trial example artifacts


In [8]:
import json

trial_results.to_csv("output/trial_summaries.csv", index=None)
with open("output/trial_summarization_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
qc_report.to_csv("output/trial_qc_report.csv", index=None)

## 2. Patient summarization walkthrough


### Read in data

In [ ]:
notes_path = "data/2026_01_16_dummy_notes.csv"
notes = pd.read_csv(notes_path)
notes.head()

### Run `summarize_patients`

The default local example is shown below. To use remote mode, pass the same remote-enabled `config` pattern shown in the trial walkthrough.


In [14]:
from matchminer_ai.patients import summarize_patients

patient_summaries, metadata, qc_report = summarize_patients(
    notes, return_metadata=True, return_qc=True
)

/home/sabrina/mmai-package/src/matchminer_ai/patients/prepare.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  normalized["note_date"] = pd.to_datetime(normalized["note_date"])


INFO 04-24 19:53:19 [utils.py:233] non-default args: {'max_model_len': 100000, 'disable_log_stats': True, 'model': 'openai/gpt-oss-120b'}
INFO 04-24 19:53:20 [model.py:549] Resolved architecture: GptOssForCausalLM


Parse safetensors files: 100%|██████████| 15/15 [00:00<00:00, 82.97it/s]

INFO 04-24 19:53:20 [model.py:1678] Using max model len 100000
INFO 04-24 19:53:20 [config.py:131] Overriding max cuda graph capture size to 1024 for performance.


(EngineCore pid=109027) INFO 04-24 19:53:20 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='openai/gpt-oss-120b', speculative_config=None, tokenizer='openai/gpt-oss-120b', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=100000, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=mxfp4, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='openai_gptoss', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, col

(EngineCore pid=109027) <frozen importlib._bootstrap_external>:1329: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=109027) <frozen importlib._bootstrap_external>:1329: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/15 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:   7% Completed | 1/15 [00:01<00:17,  1.26s/it]
Loading safetensors checkpoint shards:  13% Completed | 2/15 [00:02<00:15,  1.21s/it]
Loading safetensors checkpoint shards:  20% Completed | 3/15 [00:03<00:14,  1.25s/it]
Loading safetensors checkpoint shards:  27% Completed | 4/15 [00:04<00:13,  1.22s/it]
Loading safetensors checkpoint shards:  33% Completed | 5/15 [00:06<00:12,  1.24s/it]
Loading safetensors checkpoint shards:  40% C

(EngineCore pid=109027) INFO 04-24 19:53:42 [default_loader.py:384] Loading weights took 18.37 seconds
(EngineCore pid=109027) INFO 04-24 19:53:42 [mxfp4.py:836] Using MoEPrepareAndFinalizeNoDPEPModular
(EngineCore pid=109027) INFO 04-24 19:53:46 [gpu_model_runner.py:4820] Model loading took 65.96 GiB memory and 23.159067 seconds
(EngineCore pid=109027) INFO 04-24 19:53:52 [backends.py:1051] Using cache directory: /home/sabrina/.cache/vllm/torch_compile_cache/490ff7c3fe/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=109027) INFO 04-24 19:53:52 [backends.py:1111] Dynamo bytecode transform time: 4.72 s
(EngineCore pid=109027) INFO 04-24 19:53:55 [backends.py:285] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 2.592 s
(EngineCore pid=109027) INFO 04-24 19:53:55 [decorators.py:305] Directly load AOT compilation from path /home/sabrina/.cache/vllm/torch_compile_cache/torch_aot_compile/00d2ada678027974f610c7eadc2514e5a4b3fd22316fc22d370195cfc

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 83/83 [00:11<00:00,  7.54it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:04<00:00,  8.24it/s]


(EngineCore pid=109027) INFO 04-24 19:54:16 [gpu_model_runner.py:6046] Graph capturing finished in 16 secs, took 1.08 GiB
(EngineCore pid=109027) INFO 04-24 19:54:16 [gpu_worker.py:597] CUDA graph pool memory: 1.08 GiB (actual), 0.96 GiB (estimated), difference: 0.12 GiB (11.0%).
(EngineCore pid=109027) INFO 04-24 19:54:16 [core.py:283] init engine (profile, create kv cache, warmup model) took 29.27 seconds


Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.20s/it, est. speed input: 1622.29 toks/s, output: 120.37 toks/s]


### View results

In [15]:
patient_summaries

,patient_id,last_note_date,cancer_history_summary,general_exclusion_criteria_evidence
0,-1,2025-06-25,Age: 64\nSex: Male\nCancer type: Gastroesophag...,No evidence of common exclusion criteria – no ...
1,-10,2025-06-25,Age: 55\nSex: Female\nCancer type: Ovary (low‑...,No evidence of common boilerplate exclusion cr...
2,-11,2025-06-25,Age: 68 \nSex: Male \nCancer type: Cutaneous...,"No uncontrolled brain metastases, no active in..."
3,-12,2025-06-25,Age: 19 \nSex: Male \nCancer type: Osteogeni...,No evidence of typical trial exclusion factors...
4,-2,2025-06-25,Age: 34\nSex: Male\nCancer type: Appendiceal w...,No evidence of common trial exclusion criteria...
5,-3,2025-06-25,Age: 66\nSex: Male\nCancer type: Right breast ...,No evidence of common boilerplate exclusion cr...
6,-4,2025-06-25,Age: 62\nSex: Female\nCancer type: Glioblastom...,Presence of hypertension with persistent systo...
7,-5,2025-06-25,Age: 45\nSex: Female\nCancer type: Breast canc...,No evidence of common boilerplate exclusion cr...
8,-6,2025-06-25,Age: 68\nSex: Male\nCancer type: Small‑cell lu...,No evidence of commonly cited trial exclusion ...
9,-7,2025-06-25,Age: 70\nSex: Male\nCancer type: Prostate canc...,No evidence of commonly excluded conditions: n...


Metadata contains a snapshot of the config used and metadata on the models used. 

In [16]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 10000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9},
   'patient': {'max_model_len': 100000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'max_concurrent_requests': 16,
   'request_timeout': 600,
   'max_retries': 6,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'model_name': 'openai/gpt-oss-120b',
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.3},
   'prompt_files': {'primer': 'trial.user.primer.txt',
    'question': 'trial.user.question.txt'},
   'reasoning_marker': 'assistantfinal',
   'boilerplate_marker': '\\n.*Boilerplate.*\\n'},
  'patient': {'model_name': 'openai/gpt-oss-120b',
   'chunk_size': 10000,
   'chunk_overla

QC report on patient summarization

In [17]:
qc_report

,metric,value,denominator,percent,ids
0,patients_dropped_noninformative_summary,0,12,0.000000,[]
1,patients_exceed_embedding_token_limit,0,12,0.000000,[]
2,patients_exclusion_criteria_not_extracted,1,12,8.333333,[-9]
3,patients_missing_keyword:Age,1,12,8.333333,[-9]
4,patients_missing_keyword:Sex,1,12,8.333333,[-9]
5,patients_missing_keyword:Cancer type,1,12,8.333333,[-9]
6,patients_missing_keyword:Histology,1,12,8.333333,[-9]
7,patients_missing_keyword:Current extent,1,12,8.333333,[-9]
8,patients_missing_keyword:Biomarkers,1,12,8.333333,[-9]
9,patients_missing_keyword:Treatment history,1,12,8.333333,[-9]


Save checked-in patient example artifacts

In [18]:
patient_summaries.to_csv("output/patient_summaries.csv", index=None)
with open("output/patient_summarization_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
qc_report.to_csv("output/patient_qc_report.csv", index=None)

## 3. Embed summarized entities for matching


In [19]:
from matchminer_ai.embedding import embed_for_matching

In [20]:
# Uncomment if you want to start from previously generated summaries
# trial_results = pd.read_csv("output/trial_summaries.csv")
# patient_summaries = pd.read_csv("output/patient_summaries.csv", dtype="str")

### Trials

In [21]:
embedded_trials, metadata = embed_for_matching(
    trial_results, entity_type="trial", return_metadata=True
)

In [22]:
embedded_trials.head()

,space_trial_id,embedding
0,NCT03319901-1,"[0.0037428562063723803, 0.03316022455692291, -..."
1,NCT03319901-2,"[0.0030271741561591625, -0.0036468373145908117..."
2,NCT04792489-1,"[-0.02733524516224861, -0.03218868374824524, -..."
3,NCT04792489-2,"[0.013585780747234821, -0.03877509385347366, -..."
4,NCT04792489-3,"[-0.033121634274721146, 0.005459884647279978, ..."


Metadata contains a snapshot of the config used and metadata on the models used. 

In [23]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 10000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9},
   'patient': {'max_model_len': 100000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'max_concurrent_requests': 16,
   'request_timeout': 600,
   'max_retries': 6,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'model_name': 'openai/gpt-oss-120b',
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.3},
   'prompt_files': {'primer': 'trial.user.primer.txt',
    'question': 'trial.user.question.txt'},
   'reasoning_marker': 'assistantfinal',
   'boilerplate_marker': '\\n.*Boilerplate.*\\n'},
  'patient': {'model_name': 'openai/gpt-oss-120b',
   'chunk_size': 10000,
   'chunk_overla

### Patients

In [24]:
embedded_patients, metadata = embed_for_matching(
    patient_summaries, entity_type="patient", return_metadata=True
)

In [25]:
embedded_patients.head()

,patient_id,embedding
0,-1,"[0.05076678469777107, -0.006109500303864479, -..."
1,-10,"[-0.03486933931708336, -0.029242075979709625, ..."
2,-11,"[0.010535252280533314, 0.021748138591647148, -..."
3,-12,"[-0.00629879254847765, -0.0034406923223286867,..."
4,-2,"[-0.005802038591355085, -0.02732745185494423, ..."


Metadata contains a snapshot of the config used and metadata on the models used. 

In [26]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 10000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9},
   'patient': {'max_model_len': 100000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'max_concurrent_requests': 16,
   'request_timeout': 600,
   'max_retries': 6,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'model_name': 'openai/gpt-oss-120b',
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.3},
   'prompt_files': {'primer': 'trial.user.primer.txt',
    'question': 'trial.user.question.txt'},
   'reasoning_marker': 'assistantfinal',
   'boilerplate_marker': '\\n.*Boilerplate.*\\n'},
  'patient': {'model_name': 'openai/gpt-oss-120b',
   'chunk_size': 10000,
   'chunk_overla

In [43]:
# Optional local export if you want to reuse embeddings without rerunning this step.
embedded_patients.to_parquet("output/embedded_patients.parquet")
embedded_trials.to_parquet("output/embedded_trials.parquet")

## 4. Generate candidate matches in both directions

`generate_candidate_matches(...)` is symmetric: whichever dataframe is passed as `query_df` becomes the perspective used for ranking.


In [28]:
from matchminer_ai.matching import generate_candidate_matches

In [29]:
# Uncomment if you saved embeddings locally and want to start from them
# embedded_patients = pd.read_parquet('output/embedded_patients.parquet')
# embedded_trials = pd.read_parquet('output/embedded_trials.parquet')

### Patient-centric matching

Patients are the queries and trial spaces are the corpus.


In [30]:
patient_centric_candidate_matches = generate_candidate_matches(
    query_df=embedded_patients, corpus_df=embedded_trials
)

In [31]:
patient_centric_candidate_matches.head()

,patient_id,space_trial_id,similarity_score,rank
0,-1,NCT06293898-8,0.606075,1
1,-1,NCT05372614-2,0.585981,2
2,-1,NCT06586957-4,0.585624,3
3,-1,NCT06586957-5,0.579248,4
4,-1,NCT05482893-2,0.557887,5


### Trial-centric matching

Trial spaces are the queries and patients are the corpus.


In [32]:
trial_centric_candidate_matches = generate_candidate_matches(
    query_df=embedded_trials, corpus_df=embedded_patients
)

In [33]:
trial_centric_candidate_matches.head()

,space_trial_id,patient_id,similarity_score,rank
0,NCT03319901-1,-5,-0.069697,1
1,NCT03319901-1,-11,-0.098694,2
2,NCT03319901-1,-7,-0.109262,3
3,NCT03319901-1,-6,-0.169137,4
4,NCT03319901-1,-3,-0.171817,5


## 5. Patient-centric match quality check

The match-quality and exclusion-criteria checkers are currently set up for patient-centric candidate matches only.


In [34]:
from matchminer_ai.matching import score_match_quality

Add back the patient and trial summary context required by the patient-centric match-quality checker.


In [35]:
patient_centric_candidate_matches_with_context = (
    patient_centric_candidate_matches.merge(
        patient_summaries[["patient_id", "cancer_history_summary"]],
        on="patient_id",
        how="left",
    ).merge(
        trial_results[["space_trial_id", "clinical_space_summary"]],
        on="space_trial_id",
        how="left",
    )
)
patient_centric_candidate_matches_with_context.head()

,patient_id,space_trial_id,similarity_score,rank,cancer_history_summary,clinical_space_summary
0,-1,NCT06293898-8,0.606075,1,Age: 64\nSex: Male\nCancer type: Gastroesophag...,Age range allowed: ≥18 years. Sex allowed: Mal...
1,-1,NCT05372614-2,0.585981,2,Age: 64\nSex: Male\nCancer type: Gastroesophag...,Age range allowed: >=18. Sex allowed: Any. Can...
2,-1,NCT06586957-4,0.585624,3,Age: 64\nSex: Male\nCancer type: Gastroesophag...,Age range allowed: >=18 years. Sex allowed: bo...
3,-1,NCT06586957-5,0.579248,4,Age: 64\nSex: Male\nCancer type: Gastroesophag...,Age range allowed: >=18 years. Sex allowed: bo...
4,-1,NCT05482893-2,0.557887,5,Age: 64\nSex: Male\nCancer type: Gastroesophag...,Age range allowed: ≥18. Sex allowed: Both. Can...


Run the patient-centric match-quality checker.


In [36]:
patient_centric_match_quality_matches, metadata = score_match_quality(
    patient_centric_candidate_matches_with_context, return_metadata=True
)

Loading weights: 100%|██████████| 174/174 [00:00<00:00, 4734.87it/s]


In [37]:
patient_centric_match_quality_matches.head()

,patient_id,space_trial_id,match_quality_score,match_quality_pass
0,-1,NCT06293898-8,0.996228,True
1,-1,NCT05372614-2,0.981073,True
2,-1,NCT06586957-4,0.743658,True
3,-1,NCT06586957-5,0.736621,True
4,-1,NCT05482893-2,0.806649,True


In [38]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 10000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9},
   'patient': {'max_model_len': 100000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'max_concurrent_requests': 16,
   'request_timeout': 600,
   'max_retries': 6,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'model_name': 'openai/gpt-oss-120b',
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.3},
   'prompt_files': {'primer': 'trial.user.primer.txt',
    'question': 'trial.user.question.txt'},
   'reasoning_marker': 'assistantfinal',
   'boilerplate_marker': '\\n.*Boilerplate.*\\n'},
  'patient': {'model_name': 'openai/gpt-oss-120b',
   'chunk_size': 10000,
   'chunk_overla

## 6. Patient-centric exclusion criteria check


In [39]:
# limit to unique patient-trial ID pairs
patient_centric_match_quality_matches["trial_id"] = (
    patient_centric_match_quality_matches["space_trial_id"].str.split("-").str[0]
)
unique_patient_trial_pairs = patient_centric_match_quality_matches[
    ["patient_id", "trial_id"]
].drop_duplicates()

# add in exclusion criteria for patients and trials
patient_trial_pairs_with_exclusion_context = unique_patient_trial_pairs.merge(
    patient_summaries[["patient_id", "general_exclusion_criteria_evidence"]],
    on="patient_id",
    how="left",
).merge(
    trial_results[["trial_id", "general_exclusion_criteria"]].drop_duplicates(),
    on="trial_id",
    how="left",
)
patient_trial_pairs_with_exclusion_context.head()

,patient_id,trial_id,general_exclusion_criteria_evidence,general_exclusion_criteria
0,-1,NCT06293898,No evidence of common exclusion criteria – no ...,Severe heart disease; prolonged QT interval (>...
1,-1,NCT05372614,No evidence of common exclusion criteria – no ...,history of interstitial lung disease or pneumo...
2,-1,NCT06586957,No evidence of common exclusion criteria – no ...,* Presence of a locally advanced solid tumor a...
3,-1,NCT05482893,No evidence of common exclusion criteria – no ...,History of non‑infectious pneumonitis requirin...
4,-1,NCT06244485,No evidence of common exclusion criteria – no ...,Uncontrolled or significant cardiovascular dis...


In [40]:
from matchminer_ai.matching import exclusion_criteria_check

# run exclusion criteria check
exclusion_results, metadata = exclusion_criteria_check(
    patient_trial_pairs_with_exclusion_context, return_metadata=True
)
exclusion_results.head()

Loading weights: 100%|██████████| 174/174 [00:00<00:00, 4811.63it/s]


,patient_id,trial_id,exclusion_score,exclusion_criteria_pass
0,-1,NCT06293898,0.999754,True
1,-1,NCT05372614,0.999989,True
2,-1,NCT06586957,0.999985,True
3,-1,NCT05482893,0.999598,True
4,-1,NCT06244485,0.999993,True


In [41]:
exclusion_results["exclusion_criteria_pass"].value_counts()

exclusion_criteria_pass
True     160
False      1
Name: count, dtype: int64

In [42]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 10000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9},
   'patient': {'max_model_len': 100000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'max_concurrent_requests': 16,
   'request_timeout': 600,
   'max_retries': 6,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'model_name': 'openai/gpt-oss-120b',
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.3},
   'prompt_files': {'primer': 'trial.user.primer.txt',
    'question': 'trial.user.question.txt'},
   'reasoning_marker': 'assistantfinal',
   'boilerplate_marker': '\\n.*Boilerplate.*\\n'},
  'patient': {'model_name': 'openai/gpt-oss-120b',
   'chunk_size': 10000,
   'chunk_overla